# WS 12.2: From Pixels to Digits

You just watched a neural network recognize handwritten digits: 784 pixels go in, 10 numbers come out, and the brightest one is the network's guess. The video used two hidden layers of 16 neurons each.

Today you'll build that exact network in Python — same data, same shape — and see what it gets right, what it gets wrong, and how **precision** and **recall** work in this situation. One new idea: every classifier you've built so far picked between **two** labels (default vs. not-default, sick vs. healthy). This one picks between **ten**. The core ideas — accuracy, confusion matrix, precision, recall — still work. They just get bigger! 🐈➡️🦁

> **I will not use AI tools on this worksheet.**
>
> **Name:** \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_

### Setup

Run this cell to load libraries and a helper.

In [ ]:
#@title Setup — run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

def show_digit(image_row, label=None, predicted=None):
    """Display one MNIST digit. image_row is a length-784 array."""
    plt.imshow(image_row.reshape(28, 28), cmap="gray_r")
    plt.axis("off")
    title = ""
    if label is not None:
        title += f"actual: {label}  "
    if predicted is not None:
        title += f"predicted: {predicted}"
    if title:
        plt.title(title)
    plt.show()

---

## Part 1: The data

MNIST, which we saw in the video, is a classic dataset of 70,000 handwritten digit images. Each image is 28 × 28 pixels. We flatten each image into a row of 784 numbers — one per pixel, from 0 (black) to 255 (white) — exactly like the input layer in the video.

In [ ]:
#@title Load MNIST — run this cell (takes ~30 seconds the first time)
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="liac-arff")
X_all = mnist.data / 255.0      # scale pixels to 0-1
y_all = mnist.target.astype(int)
print("Total images:", X_all.shape[0])
print("Pixels per image:", X_all.shape[1])

Now split the data into a training set and a test set — the same `train_test_split` pattern you used in WS 12.1. We'll use 10,000 images for training and 2,000 for testing: plenty of data, and fast to train.

In [ ]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, train_size=10000, test_size=2000
)
print("Training images:", X_train.shape[0])
print("Test images:    ", X_test.shape[0])

Look at a few images. The helper `show_digit(image_row, label=...)` displays one digit and prints its label.

In [ ]:
#@title Look at some training examples — run this cell
for i in [0, 1, 2, 3, 4]:
    show_digit(X_train[i], label=y_train[i])

**Exercise 1.1:** Pick any index between 5 and 9999 and display that training image. What digit is it? Does the label match what you see?

> **Hint:** Call `show_digit(X_train[_____], label=y_train[_____])`.

In [ ]:
# Your code here

*Your answer here.*

---

## Part 2: Train the video's network — quickly

Now we build the network from the video: **two hidden layers, 16 neurons each**.

Remember from the video: a **neuron** is one of the circles that holds a number between 0 and 1 — its "activation." The **hidden** layers are just the middle layers, between the 784-pixel input layer and the 10-digit output layer. So `hidden_layer_sizes=(16, 16)` means: 16 neurons in the first hidden layer, then 16 more in the second, before reaching the output.

```python
MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=N)
```

`max_iter` is how many passes through the training data the network gets. More passes = more learning (up to a point).

First, let's train the network for only **2 passes** — barely any learning. We'll call this the **quick** network (as in "quickly trained").

Here are the four training steps, one per line:

In [ ]:
# Set up the network: 2 hidden layers of 16 neurons, 2 training passes
nn_quick = MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=2)

# Train it on the training images and labels
nn_quick.fit(X_train, y_train)

# Use the trained network to predict labels for the test images
quick_pred = nn_quick.predict(X_test)

# How often did it match the true label?
quick_acc = (quick_pred == y_test).mean()
print("Quick network test accuracy:", round(quick_acc, 3))

### Look at one digit: the number 3

Let's focus on how well the network handles the digit **3**. We'll compute precision and recall for *just that one class*, the same way you've always computed them — but now "positive" means "the model says 3" and "negative" means "the model says anything else."

Two pieces of Python to notice in the code below:

- `~` means "**not**." So `~predicted_3` is `True` for every test image where the model did **not** say 3.
- As a reminder, `.sum()` counts how many `True` values are in a boolean array. It's how we turn "which rows match" into a count of rows.

> **Precision and recall for one class:**
>
> - **Precision for one digit** = of all images the model *labeled* as that digit, how many actually were?
> - **Recall for one digit** = of all images that actually were that digit, how many did the model *find*?
>
> ```python
> predicted_this = (predictions == _____)   # model said "this digit is ____"
> actual_this    = (actual == _____)        # truly, this digit was _____
>
> tp = (predicted_this & actual_this).sum()   # said yes, was yes
> fp = (predicted_this & ~actual_this).sum()  # said yes, wasn't
> fn = (~predicted_this & actual_this).sum()  # missed one
>
> precision = tp / (tp + fp)
> recall    = tp / (tp + fn)
> ```

**Exercise 2.1:** Fill in the pattern above to compute the precision and recall for the digit **3** using the quick network's test predictions (`quick_pred`) and the true labels (`y_test`). Instead of `predicted_this` and `actual_this`, use the digit (e.g., `predicted_3` and `actual_3`). Print both.

In [ ]:
# Your code here

---

## Part 3: Train the network longer

Now train the *same architecture* for **20 passes** instead of 2. We'll call this the **full** network (trained for a longer time).

**Exercise 3.1:** Copy the four training steps from Part 2 into the cell below, but change three things:

- rename `nn_quick` → `nn_full` and `quick_pred` → `full_pred` and `quick_acc` → `full_acc`
- change `max_iter=2` to `max_iter=20`
- update the `print(...)` label to say "Full network test accuracy"

(This cell will take ~20 seconds to run.)

In [ ]:
# Your code here

**Exercise 3.2:** Copy your code from Exercise 2.1 and change it to use `full_pred` instead of `quick_pred`. Compute precision and recall for the digit **3** on the full network.

In [ ]:
# Your code here

**Exercise 3.3:** How did precision and recall for the digit 3 change between the quick and full networks? What does that tell you about what training actually does?

*Your answer here.*

### A mistake the quick network made

In [ ]:
#@title A digit the quick network got wrong — run this cell
wrong = (quick_pred != y_test)
i = np.where(wrong)[0][0]   # first mistake
show_digit(X_test[i], label=y_test[i], predicted=quick_pred[i])
print("Full network says:", full_pred[i])

**Exercise 3.4:** Describe the mistake the quick network made: what digit was it actually, and what did the quick network predict? Then say what the full network predicted — did it fix the mistake?

*Your answer here.*

---

## Part 4: Which digits does it confuse?

You just saw one specific mistake the quick network made — one 3 that got called something else. But the full network is still imperfect: it gets some test images wrong, and we'd like to know *which* digits it tends to mix up. For that, we'll build the same tool you've used before — a confusion matrix — but now it has ten rows and ten columns, one for each digit.

Use the full network from here on. Rows are the true digit, columns are what the model predicted. The diagonal (true = predicted) is correct; everything off the diagonal is a mistake.

In [ ]:
#@title Confusion matrix — run this cell
cm = pd.crosstab(y_test, full_pred, rownames=["actual"], colnames=["predicted"])
cm

**Exercise 4.1:** Look at the off-diagonal cells. Name **two digit pairs** the model confuses most — for example, "the model predicts 7 when the digit is actually 1" would be row 1, column 7.

*Your answer here.*

---

## Part 5: Easy digits, hard digits

The confusion matrix suggests the network does better on some digits than others. Let's pick two digits that look very different — a **1** and an **8** — and check.

**Exercise 5.1:** Using the full network's predictions (`full_pred`), compute precision and recall for the digit **1**. Copy your code from Exercise 3.2 and change the class number.

In [ ]:
# Your code here

**Exercise 5.2:** Now do the same for the digit **8**.

In [ ]:
# Your code here

**Exercise 5.3:** Compare the two digits. Which one does the network handle better, and does your confusion matrix from Exercise 4.1 help explain why?

*Your answer here.*

---

*Worksheet created by Ethan C. Brown in collaboration with Claude Code.*